In [9]:
import boto3
import re
import json
from datetime import datetime
from dateutil.relativedelta import relativedelta
from botocore import UNSIGNED
from botocore.client import Config

In [10]:


# Set up parameters
bucket_name = "sentinel-cogs"
date_str = input("Enter a date (YYYY-MM-DD): ")
input_date = datetime.strptime(date_str, "%Y-%m-%d")
start_date = input_date - relativedelta(months=2)
end_date = input_date + relativedelta(months=2)
print(f"Folders from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}\n")


s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
response = s3.list_objects_v2(Bucket=bucket_name)

# Collecting folder paths
folders = set()
for obj in response.get('Contents', []):
    key = obj['Key']
    parts = key.split('/')
    
    
    if len(parts) > 6:
        folder_path = "/".join(parts[:7]) + "/"  
        folder_name = parts[6]

        
        m = re.search(r'(\d{8})', folder_name)
        if m:
            try:
                folder_date = datetime.strptime(m.group(1), "%Y%m%d")
            except ValueError:
                continue
        else:
            
            try:
                folder_date = datetime.strptime(parts[4] + parts[5].zfill(2) + "01", "%Y%m%d")
            except (ValueError, IndexError):
                continue

        if start_date <= folder_date <= end_date:
            folders.add(folder_path)

print("Folders within date range:")
for folder in folders:
    print(folder)


Enter a date (YYYY-MM-DD): 2019-08-15
Folders from 2019-06-15 to 2019-10-15

Folders within date range:
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190929_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_1_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_0_L2A/
sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190929_0_L2A/


In [11]:
# Initialize variables to track the folder with the lowest cloud coverage
min_cloud_cover = float('inf')
best_folder = None

for folder in folders:
    print("\nProcessing folder:", folder)
    resp = s3.list_objects_v2(Bucket=bucket_name, Prefix=folder)
    
    json_files = [obj['Key'] for obj in resp.get('Contents', []) if obj['Key'].endswith('.json')]
    
    if not json_files:
        print("  No JSON file found in folder.")
        continue

    json_key = json_files[0]
    print("  Found JSON file:", json_key)    
    try:
        obj_response = s3.get_object(Bucket=bucket_name, Key=json_key)
        json_content = obj_response['Body'].read().decode('utf-8')
        metadata = json.loads(json_content)
    except Exception as e:
        print("  Error reading JSON from folder:", e)
        continue

    # Extracting the cloud coverage 
    cloud_cover = metadata.get('properties', {}).get('eo:cloud_cover')
    if cloud_cover is None:
        print("  Cloud cover information not available.")
        continue

    print(f"  Cloud Cover: {cloud_cover}%")
    try:
        cloud_cover_val = float(cloud_cover)
    except ValueError:
        print("  Invalid cloud cover value.")
        continue


    if cloud_cover_val < min_cloud_cover:
        min_cloud_cover = cloud_cover_val
        best_folder = folder

print(f"\nFolder with least cloud coverage: {best_folder} ({min_cloud_cover}%)")




Processing folder: sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_0_L2A/
  Found JSON file: sentinel-s2-l2a-cogs/1/C/CV/2019/10/S2B_1CCV_20191009_0_L2A/S2B_1CCV_20191009_0_L2A.json
  Cloud Cover: 66.138272%

Processing folder: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190929_1_L2A/
  Found JSON file: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190929_1_L2A/S2B_1CCV_20190929_1_L2A.json
  Cloud Cover: 78.804833%

Processing folder: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_1_L2A/
  Found JSON file: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190909_1_L2A/S2B_1CCV_20190909_1_L2A.json
  Cloud Cover: 75.949973%

Processing folder: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_0_L2A/
  Found JSON file: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_0_L2A/S2B_1CCV_20190919_0_L2A.json
  Cloud Cover: 2.253081%

Processing folder: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_20190919_1_L2A/
  Found JSON file: sentinel-s2-l2a-cogs/1/C/CV/2019/9/S2B_1CCV_2019